In [186]:
import pandas as pd
import numpy as np
import plotly.express as px
import time
from datetime import datetime
from dateutil.relativedelta import relativedelta
import quantstats as qs

### Data Loading

In [187]:
file_paths = {
    'Indonesia (IDX30)': 'IDX30_Index_Past5Y.csv',
    'Malaysia (KLSE)': 'Malaysia KLSE Composite_Past5Y.csv',
    'Singapore (STI)': 'Straits Times Index_Past5Y.csv',
    'Philippines': 'Philippines Stock Exchange Composite Index_Past5Y.csv',
    'Vietnam (VN30)': 'Vietnam VN30 Index_Past5Y.csv',
    'Thailand (SET50)': 'SET50 Index_Past5Y.csv',
    'S&P500': 'S&P500_Past5Y.csv'
}

all_closes = []

for country, path in file_paths.items():
    df = pd.read_csv(path, parse_dates=['Date'], index_col='Date')
    
    if df.index.duplicated().any():
        print(f"Cleaned duplicate dates in: {country}")
        df = df[~df.index.duplicated(keep='last')]

    close_series = df['Close'].rename(country) 
    all_closes.append(close_series)

asean_combined_df = pd.concat(all_closes, axis=1)

asean_combined_df.sort_index(inplace=True)
asean_combined_df.ffill(inplace=True)
asean_filtered_df = asean_combined_df.loc['2021-07-16':].copy()
asean_filtered_df.ffill(inplace=True)
asean_filtered_df = asean_filtered_df.replace({',': ''}, regex=True).apply(pd.to_numeric, errors='coerce')
asean_filtered_df.ffill(inplace=True)

print("Filtered Data (Starting 2021-07-16):")
print(asean_filtered_df.tail())

# Define the files and their current configurations
fx_files = {
    "IDR_USD.csv": {"has_close": False},
    "PHP_USD.csv": {"has_close": True},
    "VND_USD.csv": {"has_close": True},
    "THB_USD.csv": {"has_close": True},
    "MYR_USD.csv": {"has_close": True},
    "SGD_USD.csv": {"has_close": True}
}

# 1. Added the "flip" key to your configuration dictionary
fx_files = {
    "IDR_USD.csv": {"has_close": False, "flip": True},
    "PHP_USD.csv": {"has_close": True, "flip": False},
    "VND_USD.csv": {"has_close": True, "flip": False},
    "THB_USD.csv": {"has_close": True, "flip": False},
    "MYR_USD.csv": {"has_close": True, "flip": False},
    "SGD_USD.csv": {"has_close": True, "flip": False}
}

cleaned_dfs = {}

for filename, config in fx_files.items():
    # Load the CSV file
    df = pd.read_csv(filename)
    
    # Filter columns and rename 'Close' to 'Price' if necessary
    if config["has_close"]:
        df = df[['Date', 'Close']].rename(columns={'Close': 'Price'})
    else:
        df = df[['Date', 'Price']]
        
    # --- THE EXTRA BIT ---
    # If the flip flag is True, reverse the dataframe rows vertically
    if config.get("flip"):
        df = df.iloc[::-1].reset_index(drop=True)
    # ----------------------
        
    # Save the cleaned DataFrame to our dictionary
    cleaned_name = f"cleaned_{filename}"
    cleaned_dfs[cleaned_name] = df
    
    # Export the cleaned version back to a fresh CSV file
    df.to_csv(cleaned_name, index=False)


Filtered Data (Starting 2021-07-16):
            Indonesia (IDX30)  Malaysia (KLSE)  Singapore (STI)  Philippines  \
Date                                                                           
2026-06-07             338.99          1683.63          5025.80      5910.06   
2026-06-14             344.75          1712.03          5192.70      6135.35   
2026-06-21             331.32          1667.74          5191.73      6072.24   
2026-06-28             329.10          1679.05          5244.29      6188.03   
2026-07-05             332.65          1677.64          5433.88      6223.87   

            Vietnam (VN30)  Thailand (SET50)   S&P500  
Date                                                   
2026-06-07         1944.36           1026.89  7431.46  
2026-06-14         1963.57           1023.25  7500.58  
2026-06-21         2008.57           1011.23  7354.02  
2026-06-28         2002.56           1059.71  7483.24  
2026-07-05         1987.11           1057.12  7543.64  


## Comparative Analysis

In [188]:
# 1. Normalize the prices to base 100 starting from the first row (2021-07-16)
# This allows us to compare apples-to-apples percentage performance
normalized_df = (asean_filtered_df / asean_filtered_df.iloc[0]) * 100

# 2. Restructure the data for Plotly (melting from 'wide' to 'long' format)
# This turns the country columns into a single 'Country' column for the legend
df_melted = normalized_df.reset_index().melt(
    id_vars=['Date'], 
    var_name='Country', 
    value_name='Performance (Base 100)'
)

# 3. Create the interactive line chart
fig = px.line(
    df_melted, 
    x='Date', 
    y='Performance (Base 100)', 
    color='Country', 
    title='ASEAN Major Indices Comparison (Base 100 on 2021-07-16)',
    labels={'Date': 'Date', 'Performance (Base 100)': 'Relative Performance'}
)

# 4. Polish the layout for better readability
fig.update_layout(
    hovermode="x unified", # Shows a single tooltip line for all countries when hovering
    legend_title_text='Country',
    template='plotly_white' # Gives the chart a clean, professional look
)

# 5. Show the chart in your browser or notebook
fig.show()

In [189]:
# 1. Enforce a clean, sorted DatetimeIndex on your equity DataFrame
dollar_adjusted_asean_indices_df = asean_filtered_df.copy()
dollar_adjusted_asean_indices_df.index = pd.to_datetime(dollar_adjusted_asean_indices_df.index)
dollar_adjusted_asean_indices_df = dollar_adjusted_asean_indices_df.sort_index()

# 2. FIXED: Column keys now perfectly match the labels from Cell 2
# 3. FIXED: Added 'date_format' to handle the unique IDR structure explicitly
fx_mappings = {
    'Indonesia (IDX30)':  {'file': 'cleaned_IDR_USD.csv', 'date_format': '%d/%m/%Y', 'operation': 'multiply'},
    'Philippines':        {'file': 'cleaned_PHP_USD.csv', 'date_format': '%m/%d/%Y', 'operation': 'divide'},
    'Vietnam (VN30)':     {'file': 'cleaned_VND_USD.csv', 'date_format': '%m/%d/%Y', 'operation': 'divide'},
    'Thailand (SET50)':   {'file': 'cleaned_THB_USD.csv', 'date_format': '%m/%d/%Y', 'operation': 'divide'},
    'Malaysia (KLSE)':    {'file': 'cleaned_MYR_USD.csv', 'date_format': '%m/%d/%Y', 'operation': 'divide'},
    'Singapore (STI)':    {'file': 'cleaned_SGD_USD.csv', 'date_format': '%m/%d/%Y', 'operation': 'divide'}
}

# 4. Process and apply each exchange rate safely
for country, config in fx_mappings.items():
    # Load raw FX data and apply its explicit date format rule
    fx_df = pd.read_csv(config['file'].replace('cleaned_', '')) # Read original raw files to ensure clean start
    fx_df['Date'] = pd.to_datetime(fx_df['Date'], format=config['date_format'])
    fx_df = fx_df.set_index('Date').sort_index()
    
    # Determine the rate column dynamically (IDR uses 'Price', others use 'Close')
    rate_col = 'Price' if 'IDR' in config['file'] else 'Close'
    
    # Clean up string commas/spaces from FX data
    fx_df[rate_col] = fx_df[rate_col].astype(str).str.replace(',', '', regex=True).str.strip()
    fx_series = pd.to_numeric(fx_df[rate_col], errors='coerce')
    
    # Clean up string commas/spaces from Equity Index data
    dollar_adjusted_asean_indices_df[country] = (
        dollar_adjusted_asean_indices_df[country]
        .astype(str)
        .str.replace(',', '', regex=True)
        .str.strip()
    )
    dollar_adjusted_asean_indices_df[country] = pd.to_numeric(dollar_adjusted_asean_indices_df[country], errors='coerce')
    
    # Align FX dates to match equity index dates perfectly, filling weekend gaps
    fx_series_aligned = fx_series.reindex(dollar_adjusted_asean_indices_df.index).ffill().bfill()
    
    # Apply the accurate conversion math
    if config['operation'] == 'multiply':
        dollar_adjusted_asean_indices_df[country] = dollar_adjusted_asean_indices_df[country] * fx_series_aligned
    elif config['operation'] == 'divide':
        dollar_adjusted_asean_indices_df[country] = dollar_adjusted_asean_indices_df[country] / fx_series_aligned

print("✓ Fixed all column matching, alignment, and date format errors successfully!")

✓ Fixed all column matching, alignment, and date format errors successfully!


In [190]:
# Base 100 normalization using the USD-adjusted values
usd_normalized_df = (dollar_adjusted_asean_indices_df / dollar_adjusted_asean_indices_df.iloc[0]) * 100

# Melt wide dataframe structure to long format for Plotly
df_usd_melted = usd_normalized_df.reset_index().melt(
    id_vars=['Date'], 
    var_name='Asset', 
    value_name='USD Performance (Base 100)'
)

fig = px.line(
    df_usd_melted, 
    x='Date', 
    y='USD Performance (Base 100)', 
    color='Asset', 
    title='ASEAN Indices vs S&P 500 (USD-Adjusted, Base 100)',
    labels={'Date': 'Date', 'USD Performance (Base 100)': 'Relative Performance (USD)'}
)

fig.update_layout(hovermode="x unified", template='plotly_white')
fig.show()

In [191]:
# 1. Convert raw prices into daily percentage returns
# This measures how the markets move relative to yesterday, stripping out long-term price drift.
daily_returns = asean_filtered_df.pct_change().dropna()

# 2. Compute the Pearson correlation matrix
corr_matrix = daily_returns.corr()

# 3. Create the interactive heatmap using Plotly Express
fig = px.imshow(
    corr_matrix,
    text_auto='.2f',               # Automatically overlay the numbers rounded to 2 decimals
    color_continuous_scale='RdBu', # Red-to-Blue diverging scale (great for financial correlations)
    zmin=-1.0,                     # Correlation bottom floor
    zmax=1.0,                      # Correlation ceiling
    title='ASEAN Markets & S&P 500 Correlation Matrix (Daily Returns)',
    labels=dict(color="Correlation")
)

# 4. Clean up the layout
fig.update_layout(
    width=700,
    height=700,
    template='plotly_white'
)

# 5. Display the matrix
fig.show()

In [193]:
# Make sure quantstats has hooked into pandas
qs.extend_pandas()

# 1. Convert raw prices to daily returns
daily_returns = asean_filtered_df.pct_change().dropna()

# 2. Isolate your benchmark (assuming your S&P 500 column is named 'SP500')
benchmark_returns = daily_returns['S&P500']

# 3. Create a list of the ASEAN indices to iterate through
asean_columns = [col for col in daily_returns.columns if col != 'S&P500']

# 4. Loop through each index and compile QuantStats metrics
compiled_metrics = {}

for index in asean_columns:
    asset_returns = daily_returns[index]
    
    # FIX: Fetch the bundled greeks data array
    greeks_data = qs.stats.greeks(asset_returns, benchmark_returns)
    beta_val = greeks_data[0]   # Beta is the first item
    alpha_val = greeks_data[1]  # Alpha is the second item
    
    compiled_metrics[index] = {
        "CAGR (%)": round(qs.stats.cagr(asset_returns) * 100, 2),
        "Sharpe Ratio": round(qs.stats.sharpe(asset_returns), 2),
        "Sortino Ratio": round(qs.stats.sortino(asset_returns), 2),
        "Max Drawdown (%)": round(qs.stats.max_drawdown(asset_returns) * 100, 2),
        "Annual Volatility (%)": round(qs.stats.volatility(asset_returns) * 100, 2),
        "Beta vs S&P 500": round(beta_val, 2),
        "Excess Return vs S&P 500": round(alpha_val, 2),
    }

# 5. Turn the dictionary into a unified summary table
summary_df = pd.DataFrame(compiled_metrics).T
print(summary_df)


import warnings
import logging

                   CAGR (%)  Sharpe Ratio  Sortino Ratio  Max Drawdown (%)  \
Indonesia (IDX30)    -25.66         -0.57          -0.76            -45.80   
Malaysia (KLSE)        9.84          0.55           0.85            -14.44   
Singapore (STI)       69.61          2.12           3.17            -13.38   
Philippines           -4.43          0.04           0.06            -25.22   
Vietnam (VN30)        40.45          1.01           1.45            -39.43   
Thailand (SET50)      14.08          0.59           0.86            -33.08   

                   Annual Volatility (%)  Beta vs S&P 500  \
Indonesia (IDX30)                  38.64             0.23   
Malaysia (KLSE)                    20.81             0.20   
Singapore (STI)                    26.62             0.25   
Philippines                        34.86             0.26   
Vietnam (VN30)                     42.37             0.33   
Thailand (SET50)                   30.24             0.27   

                   Excess